In [ ]:
import pandas as pd
import random
import numpy as np
pd.set_option('display.max_columns', None)

In [ ]:
from iidsim.data.timing import avg_times
g_times = avg_times()["g"]


# Import the excel file containing P and G trains

In [ ]:
final_train_objects_filename = 'generated_schedules/p_g_ktv_psa_2days.py'


In [ ]:
df = pd.read_excel('excel_input_files_goods_sched_generation/p_g_ktv_psa_2days.xlsx')


df['direction'] = df['direction'].map({'UP': 0, 'DN': 1})
df_p = df[df['pgflag'] == 'P'].reset_index(drop=True)
print(f"Total rows: {df.shape[0]}, Passenger rows: {df_p.shape[0]}")
print(f"Passenger direction values: {df_p['direction'].unique()}")
df.tail(2)


In [ ]:
df['actualarvl_1'].dt.date.unique()

Distribution of passenger and goods trains

In [ ]:
df['pgflag'].value_counts()

In [ ]:
df_g = df[df['pgflag'] == 'G'].reset_index(drop=True)
df_g.shape

# Stations list and blocksection dictionaries

Below code finds the time required to cross the blocksection; blocksection_times

In [ ]:

import iidsim.network as network

# frozenset({stnA, stnB}) -> {'west': .., 'east': .., 'lines': [dir_mvmt, ...]}
blsec_info = {}
for b in network.blocksections_list:
    west, east = b.stn_west.name.upper(), b.stn_east.name.upper()
    key = frozenset((west, east))
    blsec_info.setdefault(key, {'west': west, 'east': east, 'lines': []})
    blsec_info[key]['lines'].append(b.dir_mvmt)

print(f"block sections loaded from network: {len(blsec_info)}")


def lines_for_travel(curr_stn, next_stn):

    info = blsec_info.get(frozenset((curr_stn, next_stn)))
    if info is None: 
        return []
    wanted_prefix = 'dn' if curr_stn == info['west'] else 'up'
    return [ln for ln in info['lines'] if ln.startswith(wanted_prefix) or ln.startswith('mid')]


def line_name(curr_stn, next_stn, line):
    """ occupancy-dict key for one physical line, e.g. 'KRDL-BCHL-dn1'."""
    info = blsec_info[frozenset((curr_stn, next_stn))]
    return f"{info['west']}-{info['east']}-{line}"


def all_line_names():
    """ replaces blocksections_list_0/1, which only ever produced one key
    per direction per adjacent pair. A section with only 'mid1' now gets
    exactly one shared timeline; a section with 'mid1' + 'mid2' gets two
    independent ones."""
    return [
        f"{info['west']}-{info['east']}-{ln}"
        for info in blsec_info.values()
        for ln in info['lines']
    ]


blec_occupied_times = {name: [] for name in all_line_names()}
print(f"physical lines tracked: {len(blec_occupied_times)}")


# Build blocksection_times (crossing time per travel direction) from the
# precomputed g_times (blocksection_times.py), instead of deriving it from
# speed x length.

blocksection_times = {**g_times['dn'], **g_times['up']}

print(f"crossing times loaded for {len(blocksection_times)} block sections (goods speeds)")


In [ ]:
len(blocksection_times)

In [ ]:
blocksection_times

In [ ]:
blocksection_times['PDT-KTV']

## sort_occupied_times()
Sorts occupied time entries in chronological order for each blocksection.

In [ ]:
def sort_occupied_times():
    #  one flat dict keyed by physical line name, so a
    # single loop covers every line regardless of how many lines a given
    # block section has.
    for key in blec_occupied_times:
        if blec_occupied_times[key]:
            blec_occupied_times[key].sort(key=lambda x: x[0])


## available_times()
Computes free time slots between occupied entries for each blocksection.


In [ ]:
def compute_schedule_bounds():
    # Schedule window bounds must come from schedule times only (schdarvl/schddprt).
    # Actual arrival/departure times are only ever used once, upstream, to seed a
    # goods train's origin (schdarvl_1 = actualarvl_1.fillna(schdarvl_1)) -- that
    # seeded value already lives in schdarvl_1 and is carried forward from here.
    time_cols = [c for c in df_combined.columns
                 if any(k in c for k in ('schdarvl', 'schddprt'))]
    all_times = pd.concat([df_combined[c] for c in time_cols]).dropna()
    return all_times.min(), all_times.max()


def available_times():
    global blec_available_times
    blec_available_times = {}

    schedule_start, schedule_end = compute_schedule_bounds()

    def compute_free_slots(occupied_times):
      
        free_times = []

        if not occupied_times:
            # Blocksection not visited yet -- the whole schedule window is free.
            if schedule_end > schedule_start:
                free_times.append([schedule_start, schedule_end - schedule_start])
            return free_times

        # Slot before the first occupied entry
        first_start = occupied_times[0][0]
        if first_start > schedule_start:
            free_times.append([schedule_start, first_start - schedule_start])

        # Gaps between consecutive trains
        for i in range(len(occupied_times) - 1):
            current_end  = occupied_times[i][1]        # arrival of current train
            next_start   = occupied_times[i + 1][0]   # departure of next train
            if next_start >= current_end:
                free_times.append([current_end, next_start - current_end])

        # Slot after the last occupied entry, up to the overall schedule end
        last_end = occupied_times[-1][1]
        if schedule_end > last_end:
            free_times.append([last_end, schedule_end - last_end])

        return free_times
    # one flat dict, one loop, keyed by physical line name (same key space as
    # blec_occupied_times).
    for key, occupied_times in blec_occupied_times.items():
        blec_available_times[key] = compute_free_slots(occupied_times)

    return blec_available_times


## goods_train(df_input)
Calculates the blec_occupied_times  (one entry per **physical line**, e.g. 'KRDL-BCHL-dn1', 'SMLG-KVLS-mid1') from all scheduled trains in df_input, then recomputes available times.

Which physical line a train's crossing is recorded against is now decided by lines_for_travel() (network topology) + first-fit.


In [ ]:
def goods_train(df_input):
    global blec_occupied_times

    #  the single per-physical-line dict.
    blec_occupied_times = {name: [] for name in all_line_names()}

    for index in range(df_input.shape[0]):
        row      = df_input.iloc[index]
        station_cols = [c for c in df_input.columns if c.startswith('station_')]
        n_stations = sum(1 for c in station_cols if pd.notna(df_input.loc[index, c]))
        row_length = 4 + n_stations * 5
        # row_length = row.count()  # count of non-null values determines actual route length

        for j in range(6, row_length, 5):
            if j + 4 >= len(df_input.columns) or j - 2 < 0:
                break

            stn1 = df_input.loc[index, df_input.columns[j - 2]]
            stn2 = df_input.loc[index, df_input.columns[j + 3]]

            # FIX: pd.isna() is reliable for both NaT and NaN
            dept_val = df_input.loc[index, df_input.columns[j]]
            arr_val  = df_input.loc[index, df_input.columns[j + 4]]
            if pd.isna(dept_val) or pd.isna(arr_val):
                continue

           
            candidates = lines_for_travel(stn1, stn2)
            if not candidates:
                continue  

            chosen = None
            for ln in candidates:
                key = line_name(stn1, stn2, ln)
                if all(arr_val <= s or dept_val >= e for s, e in blec_occupied_times[key]):
                    chosen = key
                    break
            if chosen is None:
                # shouldn't happen for real timetable data; fall back rather than drop the train
                chosen = line_name(stn1, stn2, candidates[0])

            blec_occupied_times[chosen].append([dept_val, arr_val])

    sort_occupied_times()
    available_times()


## Prepare goods train dataframe and combine with passenger trains

set the initial arrival of godos to be a actual arrival time

In [ ]:
# df_g['schdarvl_1'] = df_g['actualarvl_1'].fillna(df_g['schdarvl_1'])

In [ ]:


# Clear schdarvl/schddprt for goods trains (all stations except schdarvl_1) — actual columns are left untouched
for col in df_g.columns[6:]:
    if any(key in col for key in ['schdarvl', 'schddprt']):
        if col == 'schdarvl_1':
            continue
        df_g[col] = pd.NA

df_g1_clean = df_g.dropna(axis=1, how='all')
df_combined = pd.concat([df_p, df_g1_clean], ignore_index=True, sort=False)
print(df_combined.shape)
df_combined.tail(3)

In [ ]:
mask = df_combined['actualarvl_1'].isna() & df_combined['schdarvl_1'].notna()

df_combined.loc[mask, ['schdarvl_1', 'actualarvl_1']]

In [ ]:
df_g1_clean.tail(3)

## event_list_creation(df_combined)
Builds the initial event list — one entry per goods train representing its first unscheduled blocksection.

Each event: [trainno, row_idx, j, arrival_time, curr_stn, next_stn]


In [ ]:
event_list1 = []

def event_list_creation(df_combined):
    global event_list1
    event_list1 = []   # always reset before rebuilding

    for index, row in df_combined.iterrows():
        if row['pgflag'] == 'G':
            for j in range(6, len(df_combined.columns), 5):
                if 0 <= j - 2 < len(df_combined.columns) and j + 3 < len(df_combined.columns):
                    curr_stn     = df_combined.iloc[index, j - 2]
                    next_stn     = df_combined.iloc[index, j + 3]
                    initial_time = df_combined.iloc[index, j - 1]  # arrival at curr_stn
                    dept_time    = df_combined.iloc[index, j]       # departure (to be scheduled)

                    if pd.notna(curr_stn) and pd.notna(next_stn):
                        if pd.notna(initial_time) and pd.isna(dept_time):
                            event_list1.append([row['trainno'], index, j,
                                                initial_time, curr_stn, next_stn])
                        
                            break

## event_list_update(min_index, arr_time)
After a blocksection is scheduled, advances the event to the next blocksection,
or removes the train from the event list if it has reached its destination.



In [ ]:
def event_list_update(min_index, arr_time):
    global event_list1

    current_event = event_list1[min_index]
    row_idx   = current_event[1]
    current_j = current_event[2]
    next_j    = current_j + 5

    if next_j - 2 >= 0 and next_j + 3 < len(df_combined.columns):
        next_station = df_combined.iloc[row_idx, next_j + 3]

        if pd.notna(next_station):
            arr_time = arr_time.replace(microsecond=0)
            event_list1[min_index][3] = arr_time
            event_list1[min_index][2] = next_j
            event_list1[min_index][4] = df_combined.iloc[row_idx, next_j - 2]
            event_list1[min_index][5] = next_station
            print(f"  Event updated: train {current_event[0]}  "
                  f"{event_list1[min_index][4]} -> {next_station}  "
                  f"(ready at {arr_time})")
        else:
            print(f"  Train {current_event[0]} reached final destination.")
            event_list1.pop(min_index)
    else:
        print(f"  Train {current_event[0]} completed all stations.")
        event_list1.pop(min_index)

## scheduling_goods_time(event_list1)
Core scheduling function. Picks the goods train with the earliest pending arrival time,
finds the first available blocksection slot that fits, and writes the departure/arrival
times into `df_combined`.


In [ ]:
MIN_DWELL = pd.Timedelta(minutes=0)  # minimum station dwell before a goods train may depart

def scheduling_goods_time(event_list1):

    #  Find event with earliest arrival time
    min_value = min(event[3] for event in event_list1).replace(microsecond=0)

  
    tied = [(i, e) for i, e in enumerate(event_list1)
            if e[3].replace(microsecond=0) == min_value]
    min_index, min_event = max(tied, key=lambda x: x[1][2])

    current_time = min_value
    curr_stn     = min_event[4]
    next_stn     = min_event[5]
    block_sec    = f'{curr_stn}-{next_stn}'   # crossing-time lookup key (crossing time is per travel direction, independent of which physical line is used)
    row_idx      = min_event[1]

    # blocksection must be known
    if not (pd.notna(curr_stn) and pd.notna(next_stn)
            and block_sec in blocksection_times):
        print(f"  warning: '{block_sec}' not in blocksection_times. Removing train {min_event[0]}.")
        event_list1.pop(min_index)
        return

    if pd.isna(current_time):
        print(f"  no initial time for train at row {row_idx}. Removing.")
        event_list1.pop(min_index)
        return

    
    #  ask the network-derived lookup which physical line(s) serve
    # curr_stn -> next_stn, then pool the free slots from ALL of them. This is
    # what makes a shared single line correct: on a 'mid1'-only section a DN
    # train and an UP train now compete for the SAME slot list instead of each
    # having their own (wrong, independently tracked) availability. On a
    # section with several lines (e.g. mid1 + mid2, or dn1 + a shared mid1)
    # this gives the train its earliest slot across whichever line is free.
    line_candidates = lines_for_travel(curr_stn, next_stn)
    if not line_candidates:
        print(f"  warning: no physical line found for '{block_sec}'. Removing train {min_event[0]}.")
        event_list1.pop(min_index)
        return

    available_slots = []
    for ln in line_candidates:
        available_slots.extend(blec_available_times.get(line_name(curr_stn, next_stn, ln), []))
    available_slots.sort(key=lambda s: s[0])

    # DEBUG: show the free slots being offered to this train at decision time
    print(f"  available slots for train {min_event[0]} on '{block_sec}' "
          f"(lines: {line_candidates}): {available_slots}")

    crossing_time  = blocksection_times[block_sec]
    earliest_ready = current_time + MIN_DWELL

    scheduled = False

    for slot in available_slots:
        slot_start = slot[0]
        slot_end   = slot_start + slot[1]

        # Determine departure time within this slot
        if earliest_ready <= slot_start:
            dept_time = slot_start       # train waits for slot to open
        elif earliest_ready < slot_end:
            dept_time = earliest_ready   # train departs as soon as dwell is done
        else:
            continue                     # slot already passed, try next

        # Check slot has enough room for the full crossing
        if (slot_end - dept_time) >= crossing_time:
            arr_time = dept_time + crossing_time

            # Write scheduled times into df_combined
            df_combined.at[row_idx, df_combined.columns[min_event[2]]]     = dept_time
            df_combined.at[row_idx, df_combined.columns[min_event[2] + 4]] = arr_time

            # FIX 5: write destination col only inside success block (not from stale global)
            df_combined.at[row_idx, df_combined.columns[min_event[2] + 5]] = arr_time

            event_list_update(min_index, arr_time)
            scheduled = True
            break

    if not scheduled:
        print(f"  warning: No slot found for train {min_event[0]} on '{block_sec}' "
              f"after {earliest_ready}. Train removed from schedule.")
        event_list1.pop(min_index)

## Main scheduling loop


In [ ]:
# Step 1: Build occupied/available times from the combined (passenger +
# goods) schedule. NOW: uses df_combined instead of df_p -- goods trains
# contribute no occupied entries yet (their crossing times are still NaN),
# but their ready time (schdarvl_1) is already present in df_combined and
# feeds into compute_schedule_bounds() inside available_times(), so an
# early-ready goods train gets a slot even before the first passenger train.
goods_train(df_combined)

# Step 2: Build initial event list — one entry per goods train 
event_list_creation(df_combined)

iteration = 0
while event_list1:
    iteration += 1
    print(f"\n----------- Iteration {iteration} | Events remaining: {len(event_list1)} -----")

    scheduling_goods_time(event_list1)

    # Rebuild occupied/available times now that df_combined has new scheduled entries
    goods_train(df_combined)

print("\n All goods trains scheduled ")

## Inspect results

In [ ]:
df_combined.head(5)

In [ ]:
df_combined['schdarvl_1'].dt.date.value_counts() 

In [ ]:
df_combined['schddprt_2'].isna().sum()

In [ ]:
df_combined[df_combined['schddprt_3'].isna()]

In [ ]:
# Find rows where station exists but both schdarvl and schddprt are NaT
station_cols = [col for col in df_combined.columns if col.startswith('station_')]

issues = []
for col in station_cols:
    n = col.split('_')[1]  # get the number suffix
    schdarvl_col = f'schdarvl_{n}'
    schddprt_col = f'schddprt_{n}'
    
    if schdarvl_col in df_combined.columns and schddprt_col in df_combined.columns:
        mask = (
            df_combined[col].notna() &          # station name exists
            df_combined[schdarvl_col].isna() &  # but schdarvl is NaT
            df_combined[schddprt_col].isna()    # and schddprt is NaT
        )
        if mask.any():
            issues.extend(df_combined[mask]['trainno'].tolist())
            print(f"\n--- Issue at {col} (n={n}) ---")
            print(df_combined[mask][['trainid', 'trainno', col, schdarvl_col, schddprt_col]])

In [ ]:
issues_trains = list(set(issues))
issues_trains

In [ ]:
# remove all the issues_trains from df_combined to clean the data for next steps
df_combined = df_combined[~df_combined['trainno'].isin(issues_trains)].reset_index(drop=True)

In [ ]:
df_combined.head(5)

In [ ]:
# fill the arrival times as schedarvl if not there. only for actualarvl_1 from schdarvl_1
for index, row in df_combined.iterrows():
    if pd.isna(row['actualarvl_1']) and pd.notna(row['schdarvl_1']):
         df_combined.at[index, 'actualarvl_1'] = row['schdarvl_1']

In [ ]:
df_combined.head(2)

convert to the train objects

In [ ]:
def build_train_object(row, index, cons_number=80):
    stations_dict = {}
    actual_dict = {}
    station_cols = [col for col in row.index if col.startswith('station_') and pd.notna(row[col]) and row[col] != '']
    for st_col in station_cols:
        i = int(st_col.split('_')[1])
        arr_col = f"schdarvl_{i}"
        dep_col = f"schddprt_{i}"
        act_arr_col = f"actualarvl_{i}"
        act_dep_col = f"actualdprt_{i}"
        station = str(row[st_col]).lower()
        arrvl = pd.Timestamp(row[arr_col]) if pd.notna(row[arr_col]) else None
        dprt = pd.Timestamp(row[dep_col]) if pd.notna(row[dep_col]) else None
        act_arrvl = pd.Timestamp(row[act_arr_col]) if pd.notna(row[act_arr_col]) else None
        act_dprt = pd.Timestamp(row[act_dep_col]) if pd.notna(row[act_dep_col]) else None
        stations_dict[station] = [arrvl, dprt]
        actual_dict[station] = [act_arrvl, act_dprt]
    stations = list(stations_dict.keys())
    start_station = stations[0]
    end_station = stations[-1]
    train_obj = f"tr{index} = train('{row['trainno']}', '{row['pgflag'].lower()}', {stations_dict}, '{start_station}', '{end_station}', {cons_number}, {index}, {actual_dict})"
    return train_obj

In [ ]:
train_objects = [] 
for idx, row in df_combined.iterrows():
    train_objects.append(build_train_object(row, idx, np.random.choice([70, 75, 80, 85, 90]))) 


In [ ]:
train_objects[:10]  # Display the first 3 train objects for verification

In [ ]:
# Save train objects one per line in a file (train_class_intialised.py)


with open(final_train_objects_filename, 'w') as f:
    f.write('from iidsim.domain import train\n')
    f.write('from pandas import Timestamp\n')
    f.write('import copy as copy\n')
    f.write('\n')  # Add blank line for better formatting
    for obj in train_objects:
        f.write(obj + '\n')
    # write trains list at the end 
    print(' ')
    trains_list = ', '.join([f'tr{i}' for i in range(len(train_objects))])
    f.write('\n')  # Add blank line before trains list
    f.write(f'trains = [{trains_list}]\n')
# NOTE: this writes the same train(...)-literal .py format the engine used to
# consume directly. The simulator now loads schedules from JSON instead (see
# iidsim.schedules.loader) -- run scripts_migration_convert_schedules.py (or its
# logic) against this generated file to produce the corresponding
# src/iidsim/schedules/raw/<name>.json.
